# Adım 5: Özellik Mühendisliği
**Kişi 3 sorumluluğu** — `feature/ml-dashboard` branch

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, avg
from pyspark.sql.window import Window

GOLD_PATH    = './delta_lake/gold'
FEATURE_PATH = './delta_lake/features'

def create_spark():
    return (
        SparkSession.builder
        .appName('ClimateFeatureEngineering')
        .master('local[*]')
        .config('spark.sql.extensions',
                'io.delta.sql.DeltaSparkSessionExtension')
        .config('spark.sql.catalog.spark_catalog',
                'org.apache.spark.sql.delta.catalog.DeltaCatalog')
        .config('spark.jars.packages',
                'io.delta:delta-core_2.12:2.4.0')
        .getOrCreate()
    )

spark = create_spark()
spark.sparkContext.setLogLevel('WARN')
print('[Feature] Gold tablosu okunuyor...')
df = spark.read.format('delta').load(GOLD_PATH)
print(f'[Feature] {df.count():,} kayit yuklendi.')

In [ ]:
# Feature 1: Sicaklik farki (max - min)
df = df.withColumn('temp_range', col('max_temp_c') - col('min_temp_c'))

# Feature 2: Mevsim sayisal kodlama
df = df.withColumn(
    'season_num',
    when(col('season') == 'Winter', 0)
    .when(col('season') == 'Spring', 1)
    .when(col('season') == 'Summer', 2)
    .when(col('season') == 'Autumn', 3)
    .otherwise(0)
)

# Feature 3: Ay (gold katmaninda zaten mevcut)
print('Features 1-3 eklendi: temp_range, season_num, month')
df.select('station_id', 'date', 'avg_temp_c', 'temp_range', 'season_num', 'month').show(5)